# 01: Processed 00029/g0 inputs and coordinate QC

Read the supplied serial-section data, check physical/native coordinate relationships, and save coordinate diagnostics.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/03_3d").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "00029_native_v2")
OUT = ROOT / "outputs/03_3d" / RUN_ID
SEED = 29
MODE = "path"
LABEL_KEY = "cell_class"
SPATIAL_KEY = "spatial_st_corrected"
Z_KEY = "z_um"
FRAME = "00029__g0_registered"
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("SPAGC_RUN_ID must be a safe directory name")
DATA = ROOT / "data/3d/00029_g0_native"
ANCHORS = DATA / "st/anchors_celltype_domain_niche"
HE = DATA / "he"
if MODE not in ("baseline", "uot", "path"):
    raise ValueError("Unsupported run mode or model mode.")
print({"model_mode": MODE, "output": str(OUT)})
SECTION_RANGE = None
ROI_UM = None
MAX_CELLS_PER_ANCHOR = None
USE_NUCLEUS_PATH = False
if USE_NUCLEUS_PATH and MODE != "path":
    raise ValueError(
        "The optional supplied-nucleus experiment requires full mode and morphology paths."
    )
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from spagc._io import require_new
from spagc.reconstruction.coordinates import (
    NativeCropReader,
    source_to_registered_points,
)
from spagc.reconstruction.inputs import load_anchors, load_he_manifest

Read measured anchor AnnData and the native H&E manifest. Keep physical XY, physical depth and native image pixels distinct.

Read native H&E sections and resolve their supplied calibration, masks and registration metadata.

Read measured anchor sections, retain their expression and physical coordinates, and apply only the explicitly selected spatial or cell subset.

In [ ]:
anchors = load_anchors(
    ANCHORS,
    section_range=SECTION_RANGE,
    roi_um=ROI_UM,
    max_cells_per_anchor=MAX_CELLS_PER_ANCHOR,
    seed=SEED,
    label_key=LABEL_KEY,
    spatial_key=SPATIAL_KEY,
    z_key=Z_KEY,
)
table = load_he_manifest(HE, section_range=SECTION_RANGE, coordinate_frame=FRAME)
print({"anchor_sections": len(anchors), "he_sections": len(table)})

## 2. Check anchor coordinates and metadata

Summarize measured cell counts, genes, depth ranges and cell-type availability for each selected anchor.

In [ ]:
rows = []
for obj in anchors:
    xy = obj.obsm[SPATIAL_KEY]
    rows.append(
        {
            "section_id": str(obj.obs.section_id.iloc[0]),
            "z_um": float(obj.obs[Z_KEY].iloc[0]),
            "n_cells": obj.n_obs,
            "n_genes": obj.n_vars,
            "x_min": float(xy[:, 0].min()),
            "x_max": float(xy[:, 0].max()),
            "y_min": float(xy[:, 1].min()),
            "y_max": float(xy[:, 1].max()),
            "cell_type_available": LABEL_KEY in obj.obs,
        }
    )
qc = pd.DataFrame(rows)
display(qc)

## 3. Check the supplied registration transformations

Compare the saved forward chain with supplied registered coordinates, then check numerical inverse roundtrip residuals. Do not reapply the ST-only residual correction.

Read native H&E patches by numerically inverting the supplied registration chain. Apply the crop origin and pixel calibration once.

In [ ]:
registration_qc = []
for anchor in anchors:
    row = table[table.section_id == int(anchor.obs.section_id.iloc[0])].iloc[0]
    reader = NativeCropReader(row)
    chosen = np.linspace(0, anchor.n_obs - 1, min(anchor.n_obs, 64), dtype=int)
    original = anchor.obsm["spatial_he_local"][chosen]
    expected = anchor.obsm["spatial_registered"][chosen]
    mapped = source_to_registered_points(original, reader.section, reader.transforms)
    error = np.linalg.norm(mapped - expected, axis=1)
    if error.max() > 0.0001:
        raise ValueError(
            "Saved forward chain does not reproduce released ST registration."
        )
    registered = anchor.obsm[SPATIAL_KEY][chosen]
    source = reader.registered_to_source(registered)
    back = source_to_registered_points(source, reader.section, reader.transforms)
    inverse_error = float(np.linalg.norm(back - registered, axis=1).max())
    if inverse_error > 0.02:
        raise ValueError(
            "Inverse registration residual exceeds the declared tolerance."
        )
    registration_qc.append(
        {
            "section_id": int(row.section_id),
            "forward_reference_max_um": float(error.max()),
            "inverse_roundtrip_max_um": inverse_error,
            "native_mask_fraction": float(reader.center_supported(source).mean()),
            "native_mpp_um": reader.mpp,
        }
    )

## 4. Inspect the native H&E overlay

Inverse-map corrected physical ST coordinates to native image pixels and inspect a thumbnail overlay and a native morphology field of view.

In [ ]:
row = table[table.section_id == int(anchors[0].obs.section_id.iloc[0])].iloc[0]
reader = NativeCropReader(row)
xy = anchors[0].obsm[SPATIAL_KEY]
chosen = np.linspace(0, len(xy) - 1, min(len(xy), 2000), dtype=int)
source = reader.registered_to_source(xy[chosen])
pixel = reader.source_to_pixels(source)
scale = min(1.0, 1400 / max(reader.image.width, reader.image.height))
thumbnail = reader.image.resize(scale).cast("uchar")
image = np.frombuffer(thumbnail.write_to_memory(), np.uint8).reshape(
    thumbnail.height, thumbnail.width, 3
)
(fig, ax) = plt.subplots(figsize=(7, 7))
ax.imshow(image)
ax.scatter(pixel[:, 0] * scale, pixel[:, 1] * scale, s=2, c="cyan", alpha=0.6)
ax.set_title("Native level-0 H&E; inverse-mapped corrected ST coordinates")
center = np.median(xy, axis=0, keepdims=True)
patch = reader.extract_patch(reader.registered_to_source(center)[0])
print(
    {
        "he_sections": len(table),
        "native_pixel_size_um": reader.mpp,
        "forward_reference_max_um": max(
            r["forward_reference_max_um"] for r in registration_qc
        ),
        "inverse_roundtrip_max_um": max(
            r["inverse_roundtrip_max_um"] for r in registration_qc
        ),
    }
)

## 5. Save coordinate checks and diagnostic images

Save anchor, H&E and registration tables together with the overlay and example morphology patch.

In [ ]:
qc.to_csv(require_new(OUT / "anchor_qc.csv"), index=False)
table.drop(
    columns=["raw_source_path", "image_path", "mask_path", "metadata_path"],
    errors="ignore",
).to_csv(require_new(OUT / "he_qc.csv"), index=False)
pd.DataFrame(registration_qc).to_csv(
    require_new(OUT / "registration_qc.csv"), index=False
)
fig.savefig(require_new(OUT / "anchor_he_overlay.png"), dpi=140)
plt.close(fig)
patch.save(require_new(OUT / "native_uni2_patch_example.png"))